### Face 1: Calculo de variable objetivo.

| Vamos a encontrar un tiempo optimo 

Importamos librerias

In [3]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
tqdm.pandas()

In [ ]:
df = pd.read_csv('completo_clusters.csv')
def calcular_tiempo_optimo_mejorado(df):
    stats = df.groupby(['Direccion', 'cluster']).agg(
        demanda_95=('Total_Vehiculos', lambda x: x.quantile(0.95)),
        demanda_75=('Total_Vehiculos', lambda x: x.quantile(0.75)),
        demanda_50=('Total_Vehiculos', lambda x: x.quantile(0.50)),
        medio_ref=('Tiempo_Medio_s', 'mean')
    ).reset_index()

    df = df.merge(stats, on=['Direccion', 'cluster'], how='left')

    def definir_tiempo(row):
        if row['cluster'] == 0:
            t_opt = row['demanda_95'] * row['medio_ref']
        elif row['cluster'] == 1:
            t_opt = row['demanda_75'] * row['medio_ref']
        else:
            t_opt = row['demanda_50'] * row['medio_ref']
        
        return min(max(t_opt + 3, 20), 60)

    df['Tiempo_Optimo_Target'] = df.progress_apply(definir_tiempo, axis=1)

    df = df.drop(columns=['demanda_95', 'demanda_75', 'demanda_50', 'medio_ref'])
    
    return df

df = calcular_tiempo_optimo_mejorado(df)
# df.to_csv('completo_clusters_con_tiempo_optimo.csv', index=False)
display(df.head())

  0%|          | 0/3801 [00:00<?, ?it/s]

,Total_Vehiculos,Tiempo_Medio_s,Ocupacion_Espacial_%,Hora_Inicio,Hora_Fin,Dia_Semana,Direccion,move_time_s,cluster,Tiempo_Optimo_Target
0,9,3.74,20.96,07:58:35,07:59:08,4,3,33,0,38.613606
1,7,4.02,17.87,07:59:13,07:59:46,4,4,33,0,32.171637
2,3,1.90,7.46,07:59:51,08:00:24,4,2,33,1,20.000000
3,7,4.23,19.05,08:00:29,08:01:02,4,1,33,0,33.481618
4,8,1.80,20.82,08:01:07,08:01:40,4,3,33,0,38.613606
